# Loop-Engineering Clean-Room Alpha

Journey Name: Loop-Engineering Clean-Room Alpha

Journey Purpose: Prove that enforced-planning can create, verify, report, and reset a small external ecosystem without Brian-specific project inventory, paths, credentials, or policy content.

Notebook Mode: mixed

Related Docs: docs/plans/60_loop_engineering_cleanroom_alpha.md; docs/plans/61_cleanroom_deterministic_verified_loop.md; docs/evidence/plan60_cleanroom_alpha.md; docs/evidence/plan61_cleanroom_deterministic_loop.md

Related Code: enforced_planning/cleanroom_alpha.py; scripts/cleanroom_alpha.py

Related Tests: tests/test_cleanroom_alpha.py

Related Evidence: docs/evidence/plan60_cleanroom_alpha.md; docs/evidence/plan61_cleanroom_deterministic_loop.md


## Phase 1: Plan Clean Room

Purpose: Convert a root and component revision into a dry-run materialization plan.

Input -> Output: CleanroomSpec -> MaterializationPlan

Acceptance Criteria:
- Root path validation rejects workspace roots and escaping paths.
- Planned writes are explicit and machine-readable.

Status: proven

Execution Mode: live


In [1]:
# Live journey: every phase below calls the real enforced_planning.cleanroom_alpha
# implementation against a throwaway external root. No mocked values.
import sys, tempfile
from pathlib import Path

_repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "enforced_planning" / "cleanroom_alpha.py").exists())
sys.path.insert(0, str(_repo))

from enforced_planning.cleanroom_alpha import (
    CleanroomSpec, plan_cleanroom, materialize_cleanroom, verify_cleanroom,
    reset_cleanroom, run_demo_loop, verify_loop_trace, TRACE_DIRECTORY_RELATIVE_PATH,
)

workspace = Path(tempfile.mkdtemp(prefix="cleanroom-journey-"))
projects_root = workspace / "projects-root"
projects_root.mkdir()
spec = CleanroomSpec.build(
    root=workspace / "instance",
    component_revision="journey-demo",
    projects_root=projects_root,
)
materialization_plan = plan_cleanroom(spec).to_dict()
print("verdict:", materialization_plan["verdict"], "| planned writes:", len(materialization_plan["writes"]))
{"verdict": materialization_plan["verdict"],
 "first_writes": [w["relative_path"] for w in materialization_plan["writes"][:3]]}


verdict: planned | planned writes: 16


{'verdict': 'planned',
 'first_writes': ['Makefile', 'README.md', 'cleanroom.yaml']}

## Phase 2: Materialize Clean Room

Purpose: Write the synthetic fixture tree and install receipt into an external root.

Input -> Output: MaterializationPlan -> InstallReceipt

Acceptance Criteria:
- Apply writes only declared files and directories.
- Apply refuses foreign overwrites.

Status: proven

Execution Mode: live


In [2]:
install_receipt = materialize_cleanroom(spec).to_dict()
print("verdict:", install_receipt["verdict"], "| owned files:", len(install_receipt["owned_files"]))
# Note the real receipt schema: owned_files + owned_directories + owned_file_digests
{k: install_receipt[k] for k in ("operation", "instance_id", "verdict")} | {
    "owned_files_count": len(install_receipt["owned_files"]),
    "has_file_digests": bool(install_receipt["owned_file_digests"]),
}


verdict: materialized | owned files: 16


{'operation': 'apply',
 'instance_id': 'cleanroom-alpha',
 'verdict': 'materialized',
 'owned_files_count': 16,
 'has_file_digests': True}

## Phase 3: Verify Clean Room

Purpose: Check isolation, synthetic inventory, policy shape, receipts, and sentinel failures.

Input -> Output: external root plus InstallReceipt -> VerificationReport

Acceptance Criteria:
- Generated fixture passes verification.
- Negative controls fail with exact check ids.

Status: proven

Execution Mode: live


In [3]:
verification_report = verify_cleanroom(spec.root, projects_root=projects_root).to_dict()
print("verdict:", verification_report["verdict"])
{"verdict": verification_report["verdict"],
 "checks": verification_report["checks"],
 "findings": verification_report["findings"]}


verdict: pass


{'verdict': 'pass',
 'checks': ['root_external',
  'receipt_present',
  'receipt_paths_owned',
  'inventory_materialization',
  'required_files_present',
  'no_secret_sentinel',
  'no_personal_sentinels',
  'no_workspace_symlinks',
  'verifier_chain_integrity'],
 'findings': []}

## Phase 4: Reset Clean Room

Purpose: Remove receipt-owned artifacts while refusing tampered receipt paths.

Input -> Output: external root plus InstallReceipt -> ResetReport

Acceptance Criteria:
- Reset removes owned paths.
- Reset refuses paths outside the clean-room root.

Status: proven

Execution Mode: live


In [4]:
reset_report = reset_cleanroom(spec.root).to_dict()
print("verdict:", reset_report["verdict"], "| removed:", len(reset_report["removed_paths"]))
{"verdict": reset_report["verdict"], "sample_removed": reset_report["removed_paths"][:3]}


verdict: reset | removed: 30


{'verdict': 'reset',
 'sample_removed': ['.loop-engineering/state/install_receipt.json',
  'projects/hello-app/src/expected_message.txt',
  'projects/shared-lib/src/cleanroom_shared.py']}

## Phase 5: Run Deterministic Verified Loop

Purpose: Observe the known fixture failure, apply one declarative repair, rerun the independent verifier, and stop under explicit limits.

Input -> Output: external root plus LoopSpec -> LoopRunReceipt

Acceptance Criteria:
- The receipt records the initial verifier failure and final verifier success.
- Only the verifier can certify success; self-certification and exhausted budgets fail.
- Verifier and trace tampering fail exact integrity checks.
- An interruption persists a valid non-success receipt.

Status: proven

Execution Mode: live


In [5]:
# Fresh external root for the deterministic verified loop.
loop_spec = CleanroomSpec.build(
    root=workspace / "loop-instance",
    component_revision="journey-demo",
    projects_root=projects_root,
)
materialize_cleanroom(loop_spec)
loop_run_receipt = run_demo_loop(loop_spec.root).to_dict()
trace_path = loop_spec.root / TRACE_DIRECTORY_RELATIVE_PATH / f"{loop_run_receipt['run_id']}.json"
trace_check = verify_loop_trace(trace_path).to_dict()
print("loop verdict:", loop_run_receipt["verdict"],
      "| stop:", loop_run_receipt["stop_reason"],
      "| trace:", trace_check["verdict"])
{"status": loop_run_receipt["status"],
 "iterations_used": loop_run_receipt["iterations_used"],
 "verifier_verdicts": [t["verifier_verdict"] for t in loop_run_receipt["transitions"]],
 "trace_verdict": trace_check["verdict"]}


loop verdict: pass | stop: verifier_satisfied | trace: pass


{'status': 'completed',
 'iterations_used': 1,
 'verifier_verdicts': ['fail', 'pass'],
 'trace_verdict': 'pass'}